In [1]:
!pip install sentence_transformers
%pip install pyarrow
%pip install --use-pep517 annoy
%pip install tensorflow
%pip install pydot


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas

%pip uninstall -y sentence-transformers transformers huggingface_hub
%pip install sentence-transformers==2.2.2 transformers==4.30.2 huggingface_hub==0.16.4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Found existing installation: sentence-transformers 2.2.2
Uninstalling sentence-transformers-2.2.2:
  Successfully uninstalled sentence-transformers-2.2.2
Found existing installation: transformers 4.30.2
Uninstalling transformers-4.30.2:
  Successfully uninstalled transformers-4.30.2
Found existing installation: huggingface-hub 0.16.4
Uninstalling huggingface-hub-0.16.4:
  Successfully uninstalled huggingface-hub-0.16.4
Note: you may need to restart the kernel to use updated packages.
  Using cached sentence_transformers-2.2.2-py3-none-any.whl
  Using cached transformers-4.30.2-py3-none-any.whl.metadata (113 kB)
  Using cached huggingface_hub-0.16.4-py3-none-any.whl.metadata (12 kB)
Using cached transformers-4.30.2-py3-none-any.whl (7.2 MB)
Using cached huggingface_hub-0.16.4-py3-none-any.whl (268 kB)

[notice] A

In [1]:
import os
import gc
from annoy import AnnoyIndex
import pandas as pd
import glob
import numpy as np
from sentence_transformers import SentenceTransformer, util
import pandas as pd

for root, dirs, files in os.walk("."):
    for name in files:
        print(os.path.join(root, name))

./query_7.csv
./product_113.csv
./product_107.csv
./product_82.csv
./product_96.csv
./product_41.csv
./product_55.csv
./product_69.csv
./product_68.csv
./product_54.csv
./product_40.csv
./product_97.csv
./product_83.csv
./product_106.csv
./product_112.csv
./query_6.csv
./query_4.csv
./product_104.csv
./product_110.csv
./product_138.csv
./product_95.csv
./product_81.csv
./product_56.csv
./product_42.csv
./product_43.csv
./product_57.csv
./product_80.csv
./product_94.csv
./product_139.csv
./product_111.csv
./product_105.csv
./Cleaning.ipynb
./query_5.csv
./query_1.csv
./product_129.csv
./product_101.csv
./product_115.csv
./product_90.csv
./product_84.csv
./product_53.csv
./product_47.csv
./product_46.csv
./product_52.csv
./product_85.csv
./product_91.csv
./query_model.tflite
./product_114.csv
./product_100.csv
./product_128.csv
./query_2.csv
./product_116.csv
./product_102.csv
./product_87.csv
./product_93.csv
./product_78.csv
./product_44.csv
./product_50.csv
./product_51.csv
./product_

In [3]:
df_queries_table = pd.read_parquet('../shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_reduced_queries_table = df_queries_table[df_queries_table["small_version"] == 1]

df_products_table = pd.read_parquet('../shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_metaData_table = pd.read_csv("../shopping_queries_dataset/shopping_queries_dataset_sources.csv")

df_reduced_queries_table = df_reduced_queries_table[df_queries_table["product_locale"] == "us"]
df_reduced_queries_table = df_reduced_queries_table[(df_reduced_queries_table.esci_label == 'E') | (df_reduced_queries_table.esci_label == 'I')]

/tmp/ipykernel_808/4058875514.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_reduced_queries_table = df_reduced_queries_table[df_queries_table["product_locale"] == "us"]


In [4]:
df_joined = pd.merge(
    df_reduced_queries_table.head(),
    df_products_table,
    how="left",
    on=["product_id", "product_id"]
)

In [5]:
df_joined.head()

,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale_y
0,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch,us
1,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,Please check your existing tire Sidewall for t...,MaxAuto,NaN,us
2,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...,NaN,[QUALITY]: Hardened Steel-Iron construction wi...,Neiko,NaN,us
3,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...,No fuss. Just take off your old assembly and r...,Tire size:15x6.00-6 Ply: 4 Tubeless\n6x4.5 Whe...,Antego Tire & Wheel,Husqvarna Silver,us
4,21,!awnmower tires without rims,1,B077QMNXTS,us,E,1,1,train,MaxAuto 2 Pcs 16x6.50-8 Lawn Mower Tire for Ga...,<br>Tire Specifications:<br> 1. Material: Rubb...,"Set of 2 16X6.50-8, 16x6.50x8, 16-6.50-8 Lawn ...",MaxAuto,Black,us


In [7]:
def find_embeddings(lst_product_title,i, maxlen):
    # Define a list of sentences
    sentences = list(lst_product_title)[i:i+step]
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)

    return np.array(sentence_embeddings)

In [8]:
# Load a pre-trained model (you can choose from various models like BERT, RoBERTa, etc.)
model = SentenceTransformer('all-MiniLM-L6-v2')

In [9]:
# Create sentence embeddings for your sentences
sentences = ["This is an example sentence.", "Handling unknown words in embeddings is important."]
embeddings = model.encode(sentences, convert_to_tensor=True)

# The 'embeddings' variable now contains the sentence embeddings as PyTorch tensors
print(embeddings)
print(embeddings.shape)


tensor([[ 9.8125e-02,  6.7813e-02,  6.2523e-02,  9.5085e-02,  3.6648e-02,
         -3.9846e-03,  7.4776e-03, -1.3232e-02,  6.2884e-02,  2.2495e-02,
          7.2696e-02, -3.1274e-02,  4.6355e-02, -1.2554e-02,  4.7815e-02,
         -4.9103e-03,  4.9420e-02, -6.4109e-02, -9.6966e-02,  3.2889e-02,
          5.4104e-02,  3.5329e-02,  3.3051e-02,  1.4699e-02, -3.3431e-02,
         -2.5616e-02, -5.0792e-02,  7.3255e-02,  1.1027e-01, -2.9662e-02,
         -6.7557e-02, -3.0572e-02,  3.9560e-02,  4.5476e-02,  1.5996e-02,
          3.8550e-02, -1.0954e-02,  8.4836e-02, -4.4287e-02, -6.7964e-03,
          9.4256e-03,  5.0685e-05,  1.3036e-03, -1.1970e-02,  1.3645e-02,
         -8.4174e-02, -1.6515e-04,  5.4838e-03,  2.5615e-02, -3.1545e-02,
         -1.0734e-01, -4.5788e-02, -9.1175e-02, -2.5105e-03,  1.7998e-02,
          4.9402e-02,  6.1848e-03,  5.9796e-02,  2.7003e-02, -1.6122e-02,
         -1.8150e-02, -2.3635e-02, -9.4897e-02,  6.6216e-02,  1.4923e-01,
          2.4339e-02,  1.2102e-03,  6.

In [10]:
sample_size = 150000
df_queries_dataset_mini = df_reduced_queries_table.sample(sample_size, random_state = 42).reset_index(drop=True)
df_queries_dataset_mini.shape

(150000, 9)

In [11]:
product_cols = ['product_title', 'product_description', 'product_id']
df_products_dataset_mini = pd.merge(df_products_table[product_cols].drop_duplicates(), df_queries_dataset_mini[['product_id']].drop_duplicates() ,on = ['product_id'])
df_products_dataset_mini.shape

(139639, 3)

In [12]:
null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

df_products_dataset_mini['product_title'] = df_products_dataset_mini['product_title'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_description'] = df_products_dataset_mini['product_description'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_queries_dataset_mini['query'] = df_queries_dataset_mini['query'].apply(lambda x : str(x).lower() if pd.notna(x) else '')

null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

product_title              0
product_description    68703
product_id                 0
dtype: int64
product_title          0
product_description    0
product_id             0
dtype: int64


In [13]:
queries = list(df_queries_dataset_mini['query'].unique())

step = 1000
query_dim = 384

cols = ['q' + str(x) for x in list(range(0, query_dim))] + ['query']
cnt = 0

for i in range(0,len(queries),step):
    
    cnt += 1
    # Define a list of sentences
    sentences = list(queries)[i:i+step]

    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    
    df_tmp = pd.DataFrame(np.concatenate((sentence_embeddings, np.array(sentences).reshape(-1,1)), axis=1))
    
    df_tmp.columns = cols
    
    df_tmp.to_csv(
        f'query_{cnt}.csv', header = True, index = False)
    
#     if cnt == 2:
#         break
        
    print(i)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000


In [14]:
gc.collect()

0

In [15]:
lst_product_title = list(df_products_dataset_mini['product_title'])
lst_product_description = list(df_products_dataset_mini['product_description'])
lst_product_id = list(df_products_dataset_mini['product_id'])

In [16]:
step = 1000
product_dim = 384

cols = ['p' + str(x) for x in list(range(0, product_dim * 2))] + ['product_id']\

cnt = 0
for i in range(0,len(lst_product_title),step):
    cnt += 1
    product_title_embed = find_embeddings(lst_product_title, i, "s")
    product_description_embed = find_embeddings(lst_product_description, i, "")
    
    # Concatenate arrays column-wise and reshape lst_product_id to (x, 1)
    df_tmp = pd.DataFrame(np.concatenate((
        product_title_embed,
        product_description_embed,
        np.array(lst_product_id[i:i + step]).reshape(-1, 1)
    ), axis=1))
    
    
    df_tmp.columns = cols
    df_tmp.to_csv(f'product_{cnt}.csv', header = True, index = False)
    
#     if (cnt == 2):
#         break
    
    print(i)

/tmp/ipykernel_316/635942248.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(sentence_embeddings)


0


/tmp/ipykernel_316/635942248.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(sentence_embeddings)


1000


/tmp/ipykernel_316/635942248.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(sentence_embeddings)


...

137000


/tmp/ipykernel_316/635942248.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(sentence_embeddings)


138000


/tmp/ipykernel_316/635942248.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(sentence_embeddings)


139000


In [3]:
gc.collect()

60

In [18]:
# Get a list of CSV files that start with "product_"
file_list = glob.glob('product_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_product_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_product_df.to_csv('product_embeddings.csv', index=False)

print(concatenated_product_df.shape)

concatenated_product_df.head()

(277240, 771)


,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p761,p762,p763,p764,p765,p766,p767,product_id,product_title,pid
0,-0.038078,0.046183,-0.046560,-0.029900,0.033105,0.036471,0.026031,0.075297,-0.044393,0.024491,...,0.000266,0.008444,0.022443,0.025157,-0.112528,-0.013211,0.010067,B082BH7LTP,NaN,NaN
1,-0.026833,0.036172,-0.024524,-0.012489,0.041351,0.025878,0.023794,0.047958,0.000890,0.034477,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B008FOIUYQ,NaN,NaN
2,-0.027767,-0.002590,-0.024761,-0.017926,0.000495,0.021453,0.041093,0.133278,-0.064400,0.028866,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00B9RKDNW,NaN,NaN
3,0.013659,0.020149,-0.058309,-0.037144,0.019248,-0.018274,0.113505,0.064983,0.021726,0.017442,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00CF435PQ,NaN,NaN
4,-0.076317,0.100065,-0.125701,-0.031984,-0.044950,-0.039503,0.054975,0.083635,-0.067122,-0.052565,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00CJ0TZ2S,NaN,NaN


In [19]:
# Get a list of CSV files that start with "product_"
file_list = glob.glob('query_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_query_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_query_df.to_csv('query_embeddings.csv', index=False)

print(concatenated_query_df.shape)

concatenated_query_df.head()

(28394, 385)


,q0,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q375,q376,q377,q378,q379,q380,q381,q382,q383,query
0,-0.030207,0.088923,0.073030,0.110539,0.084778,0.014884,0.060988,0.006597,-0.004667,-0.128469,...,0.018233,-0.023314,-0.024556,-0.031398,-0.042695,0.061797,0.009030,-0.014782,-0.054768,morning recovery
1,-0.064102,0.023352,-0.041795,0.023195,0.081636,-0.017785,-0.005570,-0.009171,0.032116,-0.042893,...,-0.036001,0.000856,-0.057846,-0.112102,0.029485,0.031720,-0.028930,-0.015094,0.068795,lowes
2,0.013132,0.072070,-0.065992,-0.002896,-0.040129,0.013525,0.012280,-0.013594,-0.027428,0.057251,...,0.042006,0.093058,-0.018531,-0.044261,0.036163,0.088064,-0.001749,-0.007720,0.037286,task chairs without arms 300 lb
3,-0.039690,-0.034436,-0.005114,-0.038683,0.040749,-0.068795,0.187906,0.036407,-0.022499,0.017866,...,-0.132944,0.050967,0.103548,0.003425,-0.065521,-0.063869,0.011004,-0.062140,0.046877,contact lens colored for eyes
4,-0.026860,0.098137,-0.010122,-0.067484,-0.046869,-0.006085,0.006388,-0.044399,-0.057481,-0.047813,...,0.014596,0.012656,0.028947,0.140658,0.003223,-0.049562,0.096512,-0.011497,-0.064696,plates to paint and decorate


In [20]:
df_queries_dataset_mini  = pd.merge(df_queries_dataset_mini, concatenated_query_df, on = 'query')
df_queries_dataset_mini  = pd.merge(df_queries_dataset_mini, concatenated_product_df, on = 'product_id')

print(df_queries_dataset_mini.shape)

df_queries_dataset_mini = df_queries_dataset_mini.drop_duplicates()

df_queries_dataset_mini.head()

df_queries_dataset_mini.to_csv('dataset_mini.csv', header = True, index = False)

(302728, 1163)


In [6]:
df = pd.read_csv("./dataset_mini.csv")
print(df.head())
df.shape

   example_id                             query  query_id  product_id  \
0       27501  1.0 reading glasses without ears       967  B085G83F81   
1       27501  1.0 reading glasses without ears       967  B085G83F81   
2     2217414           words i could not write    113782  1975778375   
3     2217414           words i could not write    113782  1975778375   
4     1833968                simpsons bart soul     93686  B07YQB32KK   

  product_locale esci_label  small_version  large_version  split        q0  \
0             us          E              1              1  train  0.051139   
1             us          E              1              1  train  0.051139   
2             us          E              1              1  train -0.020010   
3             us          E              1              1  train -0.020010   
4             us          I              1              1   test -0.037605   

   ...      p760      p761      p762      p763      p764      p765      p766  \
0  ...      

(302726, 1163)

In [2]:
df_products_table = pd.read_parquet('../shopping_queries_dataset/shopping_queries_dataset_products.parquet')

df_product_embedding = pd.merge(
    pd.read_csv('./product_embeddings.csv'),
    df_products_table[['product_id','product_title']],
    on = ['product_id']
).reset_index(drop=True)
df_product_embedding = df_product_embedding.drop_duplicates()


df_product_embedding['pid'] = range(0, df_product_embedding.shape[0])



/tmp/ipykernel_884/1342393268.py:4: DtypeWarning: Columns (0: product_title) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('./product_embeddings.csv'),


In [ ]:

df_query_embedding = pd.read_csv('./query_embeddings.csv')
df_query_embedding = df_query_embedding.drop_duplicates()

df_query_embedding['qid'] = range(0, df_query_embedding.shape[0])
df_query_embedding.head()

In [7]:
# Remove a column named 'old_column'
# Rename 'old_name' to 'new_name'
df_product_embedding = df_product_embedding.rename(columns={'product_title_y': 'product_title'})
df_product_embedding.head()

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p761,p762,p763,p764,p765,p766,p767,product_id,pid,product_title
0,-0.038078,0.046183,-0.046560,-0.029900,0.033105,0.036471,0.026031,0.075297,-0.044393,0.024491,...,0.000266,0.008444,0.022443,0.025157,-0.112528,-0.013211,0.010067,B082BH7LTP,0,GREEN LIFESTYLE Black Bleach Proof Towels Bulk...
1,-0.026833,0.036172,-0.024524,-0.012489,0.041351,0.025878,0.023794,0.047958,0.000890,0.034477,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B008FOIUYQ,1,Utopia Towels Cotton Bleach Proof Salon Towels...
2,-0.027767,-0.002590,-0.024761,-0.017926,0.000495,0.021453,0.041093,0.133278,-0.064400,0.028866,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00B9RKDNW,2,Bio Plas 0090 Polypropylene 96 Well Reversible...
3,0.013659,0.020149,-0.058309,-0.037144,0.019248,-0.018274,0.113505,0.064983,0.021726,0.017442,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00CF435PQ,3,Tube Color Storage Rack - 3 Pack
4,-0.076317,0.100065,-0.125701,-0.031984,-0.044950,-0.039503,0.054975,0.083635,-0.067122,-0.052565,...,-0.012748,0.061454,0.035641,0.158746,0.126410,0.046549,-0.015717,B00CJ0TZ2S,4,Pana Black 6 Tier Large Wall Mounted Metal Rac...


In [9]:
print(df_query_embedding.shape, df_product_embedding.shape)

(28394, 386) (283691, 772)


In [10]:
query_tower_input_dim = 384
product_tower_input_dim = (384*2)

q = AnnoyIndex(query_tower_input_dim, 'dot')
mp_query_dict = {}


for ix,row in df_query_embedding.iterrows():
    mp_query_dict[row['qid']] = row['query']
    
    key = int(row['qid'])
    vec = list(row[['q'+str(x) for x in list(range(query_tower_input_dim))]])
    
    #     print(key,vec)
    q.add_item(key,vec)


q.build(100) # 100 trees
q.save('query.tree')

top_k = 20
mat = []
for ix,row in df_query_embedding.iterrows():
    item = row['query']
    mat.append([item] + [mp_query_dict[x] for x in q.get_nns_by_item(row['qid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['query_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

df_neighbors1 = pd.DataFrame(mat, columns = cols)

display(df_neighbors1.head(50))

,query_id,nearest_1,nearest_2,nearest_3,nearest_4,nearest_5,nearest_6,nearest_7,nearest_8,nearest_9,...,nearest_11,nearest_12,nearest_13,nearest_14,nearest_15,nearest_16,nearest_17,nearest_18,nearest_19,nearest_20
0,morning recovery,morning recovery original,ok to wake clock,c-50 blemish night treatment,i’m tired of waking up and not being in maui,booty enhancing drink,light therapy insomnia,healing depression without medication,timely alarm,sleep supplement without melatonin,...,bedtime press,alarm clock without snooze,natural sleep aid without melatonin,lin manuel miranda good morning good night,dreamstation,best sleep aid for insomnia,sleepless in southampton book,maxi nursing nightgown,joerns hospital bed,lifesaver
1,lowes,nootropics depot,parker s huntington,mainstays kitchen,marshall stanmore 2,kc mills,lawn trencher,u-line refrigerator,wrench arborist,inc,...,houses,lucinda s kitchen,pet co,lori wall bed,kitchen cabinets,nls,pipe,dyson hairdryer,taos te,profloss
2,task chairs without arms 300 lb,ergonomic desk chair no arms,lobby chairs without arms,gaming chair for adults 300lbs,upholstery arm chairs,gaming chair 300 lb weight capacity,balance chairs,low back desk chair without arms,bungee office chair without arms,lawn chair without arms,...,rolling desk chair without arms,chairs without wheels,gaming chair 500lb weight capacity,steno chair without arms,0ffice chairs without wheels,chair covers for office chairs without arms,wood desk chairs without wheels,desk chairs without wheels for kids,ergonomic office chair without wheels,home office chair without wheels
3,contact lens colored for eyes,colored contact lenses,color contact lenses,color contacts lenses for eyes non prescription,contacts lenses for eyes cosplay gray,white contact lenses,contacts lenses,halloween colored contacts,gold color contacts,light green contacts,...,blue light glasses without yellow tint,bike sunglasses night lense,canon red ring lenses,celestron lens shade,white shades glasses for women,blue light glasses kids,goggles for led light therapy,red eagle eye led lights,optcon a eye drops,under armour sunglasses for men pink
4,plates to paint and decorate,garden party plates,wood look paper plates,christmas plates,burlap look paper plates,kids porcelain plates,paper bowls and plates,ceramic plates,"7"" paper plates",pastel unicorn party plates,...,floral dessert plates,front plates for cars hi,pink shell plates,joy ceramic plates,plate for baby sheet,kids plates not plastic,plastic plates and forks,disposable plates,cow paper plate,black plate set
5,kids clothes 12 years,9 year old clothes,kid 9 clothes,fashion show clothes for kids,2 year old boys clothes,kids winter clothing,kids clothing sets for boys,teen girl clothes 13 years old,baby girl 6-9 months winter clothes,dress up clothes for little girls,...,kids dress,vintage newborn boy clothes,0-3 month boy fall clothes,cute girl outfits age 9,dinosaur clothes children,winter cozy toddler girl clothes,baby dress up clothes,long sleeve shirts for kids,art supplies for kids age 7-9,vans toddler clothes
6,ergonomic desk chair without wheels,ergonomic office chair without wheels,home office chair without wheels,desk chair without wheels cheap,small office chair without wheels,white desk chair without wheels,wood desk chairs without wheels,office chair without wheels or lift,gaming chair without wheels,adjustable chair without wheels,...,ergonomic desk,kids office chair without wheels,ergonomic desk chair no arms,office chair no wheels,desk chairs without wheels for kids,rolling desk chair without arms,ergonomic chair wooden,office chair not on wheels,desk chair for teens girls without wheels,foldable ergonomic chair
7,gacha life,gakus,ghibli,lifesaver,streigh gama,apha goc,uludag gazoz,pista,detective pikachu,zh,...,zervos,zerplus,sal and gabi break the universe,life on the line movie,grode,lunir,naruto kunai,hard zelter,lifes too short abby jimenez,pixie
8,34 curvy jeans,"stretchy skinny jeans, d

In [8]:
product_tower_input_dim = (384*2)
p = AnnoyIndex(product_tower_input_dim, 'dot')
mp_product_dict = {}

for ix,row in df_product_embedding.iterrows():
    mp_product_dict[int(row['pid'])] = row['product_title']
    
    key = int(row['pid'])
    vec = list(row[['p'+str(x) for x in list(range(product_tower_input_dim))]])
    
#     print(key,vec)
    p.add_item(key,vec)
    if ix == 50:
        break
    

p.build(100) # 100 trees
p.save('product.tree')


p = AnnoyIndex(product_tower_input_dim,  'euclidean')
p.load('product.tree')



top_k = 20
mat = []
for ix,row in df_product_embedding.iterrows():
    item = row['product_title']
    mat.append([item] + [mp_product_dict[x] for x in p.get_nns_by_item(row['pid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['product_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

print(cols)

df_neighbors2 = pd.DataFrame(mat, columns = cols)

display(df_neighbors2.head(200))

['product_id', 'nearest_1', 'nearest_2', 'nearest_3', 'nearest_4', 'nearest_5', 'nearest_6', 'nearest_7', 'nearest_8', 'nearest_9', 'nearest_10', 'nearest_11', 'nearest_12', 'nearest_13', 'nearest_14', 'nearest_15', 'nearest_16', 'nearest_17', 'nearest_18', 'nearest_19', 'nearest_20']


,product_id,nearest_1,nearest_2,nearest_3,nearest_4,nearest_5,nearest_6,nearest_7,nearest_8,nearest_9,...,nearest_11,nearest_12,nearest_13,nearest_14,nearest_15,nearest_16,nearest_17,nearest_18,nearest_19,nearest_20
0,GREEN LIFESTYLE Black Bleach Proof Towels Bulk...,"20 Pieces Hair Dye Brush and Bowl Set, Hair Dy...",Utopia Towels Cotton Bleach Proof Salon Towels...,Healthcom Set of 5 Hair Combs Set Professional...,10 Pieces Hair Barber Styling Comb Set with 10...,Binocktails Bev-Brush Paddle Hairbrush Secret ...,YJYdada 1PC Mixing Paint Stirrer Pro Salon Hai...,Volumizing Biotin Shampoo and Conditioner Set ...,6 Pieces Rat Tail Comb Fiber Teasing Combs Rat...,Krewey Hair Cutting Scissors Professional Home...,...,Biotin Shampoo and Conditioner Set for Hair Gr...,Hair Extension Holder and Hanger – Professiona...,Biotin Hair Thickening Spray for Thin Hair Tex...,Tube Color Storage Rack - 3 Pack,Clearform ML7889 Clear Acrylic Extra Large Tub...,Back Pain Relief Bolster Pillow - Half Moon Kn...,Pana Black 6 Tier Large Wall Mounted Metal Rac...,Cushy Form Bolster Pillow for Lumbar and Leg S...,Cushy Form Bolster Pillow for Lumbar and Leg S...,Hair Dye Brush Holder - Hair Color Brush Stand...
1,Utopia Towels Cotton Bleach Proof Salon Towels...,Tube Color Storage Rack - 3 Pack,Hair Dye Brush Holder - Hair Color Brush Stand...,Pana Black 6 Tier Large Wall Mounted Metal Rac...,Clearform ML7889 Clear Acrylic Extra Large Tub...,Zippity Outdoor Products ZP19018 Manchester No...,Leakproof Hidden Flask - 2 Secret Sunscreen Fl...,Backyard X-Scapes Natural Rolled Bamboo Fence ...,Bio Plas 0090 Polypropylene 96 Well Reversible...,GoPong Sneak Alcohol Anywhere Ice Flask (2 Pac...,...,"Laube Blade Case, Mini",Hallmark Shoebox Maxine All Occasions Card Ass...,"Hallmark Halloween Cards Assortment, Wicked Ca...","Hallmark Shoebox Funny Anniversary Card, Love ...","Sedona Method Course, the Vols 1-2","Hallmark Shoebox Funny Boxed Christmas Cards, ...",Hallmark Shoebox Funny Christmas Card (Bulldog...,Hallmark Shoebox Funny Birthday Card (Cold Beers),OSP Home Furnishings Wicker Papasan Chair with...,Hallmark Shoebox Funny Birthday Card (4 Out of...
2,Bio Plas 0090 Polypropylene 96 Well Reversible...,Tube Color Storage Rack - 3 Pack,Clearform ML7889 Clear Acrylic Extra Large Tub...,Pana Black 6 Tier Large Wall Mounted Metal Rac...,Leakproof Hidden Flask - 2 Secret Sunscreen Fl...,Hair Dye Brush Holder - Hair Color Brush Stand...,Utopia Towels Cotton Bleach Proof Salon Towels...,"Tuf-Tite Septic Tank Riser, 24''x12''",GoPong Sneak Alcohol Anywhere Ice Flask (2 Pac...,Zippity Outdoor Products ZP19018 Manchester No...,...,"Hallmark Shoebox Funny Anniversary Card, Love ...","Laube Blade Case, Mini",Hallmark Shoebox Funny Birthday Card (Cold Beers),Hallmark Shoebox Funny Birthday Card (Cupcake),Hallmark Shoebox Funny Birthday Card (4 Out of...,Hallmark Shoebox Maxine All Occasions Card Ass...,"Hallmark Shoebox Funny Boxed Christmas Cards, ...",Hallmark All Occasion Handmade Boxed Set of As...,Hallmark Shoebox Funny Birthday Card (A Very S...,Hallmark Shoebox Funny Christmas Card (Bulldog...
3,Tube Color Storage Rack - 3 Pack,Hair Dye Brush Holder - Hair Color Brush Stand...,Clearform ML7889 Clear Acrylic Extra Large Tub...,Bio Plas 0090 Polypropylene 96 Well Reversible...,Pana Black 6 Tier Large Wall Mounted Metal Rac...,Utopia Towels Cotton Bleach Proof Salon Towels...,GoPong Sneak Alcohol Anywhere Ice Flask (2 Pac...,Leakproof Hidden Flask - 2 Secret Sunscreen Fl...,Zippity Outdoor Products ZP19018 Manchester No...,"Tuf-Tite Septic Tank Riser, 24''x12''",...,Hallmark Shoebox Funny Christmas Cards Assortm...,Hallmark Shoebox Funny Birthday Card (4 Out of...,"Hallmark Halloween Cards Assortment, Wicked Ca...",Hallmark Shoebox Funny Birthday Card (Cold Beers),Hallmark Shoebox Funny Birthday Card (Cupcake),Hallmark Shoebox Maxine All Occasions Card Ass...,"Hallmark Shoebox Funny Boxed Christmas Cards, ...",Hallmark Shoebox Funny Birthday Card (A Very S...,H